Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

LOAD DATA

In [2]:
df = pd.read_csv(
    "../data/processed/clean_sales.csv"
)

print(df.shape)
df.head()

(9994, 27)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sales,Quantity,Discount,Profit,Year,Month,Quarter,Weekday,Profit Margin,Sales Category
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,261.9600,2,0.00,41.9136,2016.0,November,4.0,Tuesday,16.00,Medium
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,731.9400,3,0.00,219.5820,2016.0,November,4.0,Tuesday,30.00,High
2,3,CA-2016-138688,2016-06-12,NaN,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,14.6200,2,0.00,6.8714,2016.0,June,2.0,Sunday,47.00,Low
3,4,US-2015-108966,2015-10-11,NaN,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,957.5775,5,0.45,-383.0310,2015.0,October,4.0,Sunday,-40.00,High
4,5,US-2015-108966,2015-10-11,NaN,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,22.3680,2,0.20,2.5164,2015.0,October,4.0,Sunday,11.25,Low


In [3]:
df.describe()

,Row ID,Postal Code,Sales,Quantity,Discount,Profit,Year,Quarter,Profit Margin
count,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,4042.000000,4042.000000,9994.000000
mean,4997.500000,55190.379428,229.858001,3.789574,0.156203,28.656896,2015.723404,2.940376,12.031393
std,2885.163629,32063.693350,623.245101,2.225110,0.206452,234.260108,1.119750,1.044261,46.675435
min,1.000000,1040.000000,0.444000,1.000000,0.000000,-6599.978000,2014.000000,1.000000,-275.000000
25%,2499.250000,23223.000000,17.280000,2.000000,0.000000,1.728750,2015.000000,2.000000,7.500000
50%,4997.500000,56430.500000,54.490000,3.000000,0.200000,8.666500,2016.000000,3.000000,27.000000
75%,7495.750000,90008.000000,209.940000,5.000000,0.200000,29.364000,2017.000000,4.000000,36.250000
max,9994.000000,99301.000000,22638.480000,14.000000,0.800000,8399.976000,2017.000000,4.000000,50.000000


In [5]:
X = df.drop(
    columns=[
        "Sales",
        "Profit",
        "Profit Margin"
    ]
)

y = df["Sales"]

In [6]:
print(df.columns.tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Year', 'Month', 'Quarter', 'Weekday', 'Profit Margin', 'Sales Category']


In [7]:
X = X.drop(columns=["Row ID"], errors="ignore")

In [8]:
X = X.drop(columns=["Order ID"], errors="ignore")

Date Feature Engineering

In [9]:
df["Order Date"] = pd.to_datetime(df["Order Date"])

In [10]:
df["Order Year"] = df["Order Date"].dt.year

df["Order Month"] = df["Order Date"].dt.month

df["Order Day"] = df["Order Date"].dt.day

df["Order Weekday"] = df["Order Date"].dt.dayofweek

Create Order-Level Features

In [11]:
df["Discount Amount"] = (
    df["Sales"] * df["Discount"]
)

In [12]:
df["Pre Discount Sales"] = (
    df["Sales"] / (1 - df["Discount"])
)

In [13]:
df["Pre Discount Sales"] = np.where(
    df["Discount"] < 1,
    df["Sales"] / (1 - df["Discount"]),
    df["Sales"]
)

In [15]:
customer_orders = (
    df.groupby("Customer Name")["Order ID"]
    .nunique()
    .reset_index(name="Customer Orders")
)

In [16]:
df = df.merge(
    customer_orders,
    on="Customer Name",
    how="left"
)

In [17]:
customer_revenue = (
    df.groupby("Customer Name")["Sales"]
    .sum()
    .reset_index(name="Customer Total Revenue")
)

In [18]:
df = df.merge(
    customer_revenue,
    on="Customer Name",
    how="left"
)

Customer Average Order Value

In [19]:
df["Customer AOV"] = (
    df["Customer Total Revenue"]
    / df["Customer Orders"]
)

Product-Level Features

In [20]:
product_orders = (
    df.groupby("Product Name")["Order ID"]
    .nunique()
    .reset_index(name="Product Orders")
)

In [21]:
df = df.merge(
    product_orders,
    on="Product Name",
    how="left"
)

In [22]:
product_revenue = (
    df.groupby("Product Name")["Sales"]
    .sum()
    .reset_index(name="Product Total Revenue")
)

In [23]:
df = df.merge(
    product_revenue,
    on="Product Name",
    how="left"
)

In [24]:
print(df.columns.tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Year', 'Month', 'Quarter', 'Weekday', 'Profit Margin', 'Sales Category', 'Order Year', 'Order Month', 'Order Day', 'Order Weekday', 'Discount Amount', 'Pre Discount Sales', 'Customer Orders', 'Customer Total Revenue', 'Customer AOV', 'Product Orders', 'Product Total Revenue']


In [25]:
leakage_columns = [
    "Customer Total Revenue",
    "Customer AOV",
    "Product Total Revenue",
    "Pre Discount Sales"
]

df_model = df.drop(
    columns=leakage_columns,
    errors="ignore"
)

Select Features

In [26]:
features = [
    "Quantity",
    "Discount",
    "Shipping Days",
    "Order Year",
    "Order Month",
    "Order Weekday",
    "Category",
    "Sub-Category",
    "Region",
    "Segment"
]

In [28]:
numeric_features = [
    "Quantity",
    "Discount",
    "Shipping Days",
    "Order Year",
    "Order Month",
    "Order Weekday"
]

In [29]:
categorical_features = [
    "Category",
    "Sub-Category",
    "Region",
    "Segment"
]

In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

Train-Test Split

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [32]:
print("Training:", X_train.shape)

print("Testing:", X_test.shape)

Training: (7995, 22)
Testing: (1999, 22)


In [33]:
import os

os.makedirs(
    "../data/ml",
    exist_ok=True
)

In [35]:
df_model.to_csv(
    "../data/ml/ml_sales_dataset.csv",
    index=False
)